# 第 1 章　六张栅格，能不能一起建模？

> **本章问题**：甘南县的地表温度、高程、NDVI、气温、降水和土壤水分，这六张栅格能不能直接放在一起，训练一个模型？

## 地理情境

甘南县在**黑龙江省齐齐哈尔市**，地处大兴安岭东麓向松嫩平原的过渡带，县域约 122.90–124.47°E、47.59–48.53°N。打开边界文件 `GanNan.shp`，属性表里的 `name` 是“甘南县”，行政区划代码为 `230225`；这不是甘肃省甘南藏族自治州（代码 `623000`）。本课所有栅格都覆盖这一县域，目标是 1 km 左右的生长季地表温度（LST），另外五张是可能解释温度差异的地形、植被、气温、降水和土壤水分。

六张图来自不同传感器、不同处理流程，原生分辨率从二十多米到十公里不等，坐标系也不完全一样。把它们“放在一起”之前，先要回答三个朴素问题：它们是不是同一个地方？各自的像元代表什么？哪些像元能信、哪些只是填充值？

## 学完能做什么

- 用 `rasterio` 读出 GeoTIFF 的 CRS、分辨率、范围、NoData、有效像元与数值范围；
- 独立核实数据对应的是甘南县，而不是同名的“甘南”；
- 把六张不同网格的栅格画在同一页上，并说明为什么还不能直接叠成一张表；
- 识别“未声明 NoData”“边界外 0 填充”“单位缺失”这三类常见数据风险，并写进数据字典。

## 1.1　先认环境，再认数据

我们先确认一下，这个 Notebook 是不是跑在 `geoml` 里。请运行下面这格，看看 `CONDA_PREFIX` 指向哪里。它应该落在 `...\envs\geoml`。如果不是，先切回正确环境再往下做，因为 GDAL/PROJ 读坐标时可能会受环境影响。


In [ ]:
import os
import sys
import json
import time
import platform
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
from rasterio.features import geometry_mask
from rasterio.warp import transform_bounds

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
import geoml_data as gd
import geoml_style as gs

gs.setup()
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 42)

T0 = time.time()
RUN_START = datetime.now().astimezone()
print("Python:", sys.executable)
print("Conda:", os.environ.get("CONDA_PREFIX"))
print("rasterio:", rasterio.__version__, "| geopandas:", gpd.__version__)
print("项目根目录:", PROJECT)
print("运行开始:", RUN_START.isoformat(timespec="seconds"))

## 1.2　逐张登记：身份、网格、NoData

先别急着把文件叠起来。我们先给每张图做一份“身份证”：它在哪个坐标系里、像元多大、覆盖到哪里、用什么标记缺测。

下面这段只读原始元数据，不会改任何像元。读结果时请特别留意，`GNNDVI.tif` 没声明 NoData，降水和土壤水分也没声明 NoData，但数组里能看到 `NaN`。文件标签和像元内容有时不一致，后面得分开核查。


In [ ]:
FILES = [
    ("GNLST.tif", "地表温度 LST", "目标变量 y",
     "2021 年（文件名/项目说明仅到年，月日未知）",
     "ArcGIS 掩膜提取自 LST_Day2021.tif",
     "目标栅格，决定 176×106 的参考网格"),
    ("GNDEM.tif", "数字高程 DEM", "预测变量",
     "静态地形（快照时间未注明）",
     "ArcGIS 掩膜提取自 hlj_DEM.tif",
     "UTM 52N 的 26.7 m 地形，需投影到经纬度网格"),
    ("GNNDVI.tif", "归一化植被指数 NDVI", "预测变量",
     "未注明（文件生成时间不代表观测时间）",
     "元数据未给血缘",
     "无 NoData 声明；边界外大量 0 填充"),
    ("GNTEM.tif", "气温 TEM", "预测变量",
     "2021 年（文件名/项目说明仅到年）",
     "ArcGIS 掩膜提取自 TEM2021.tif",
     "与 LST 同年，但时间聚合方式未知"),
    ("precip_sum_2020.tif", "年降水 SUM", "预测变量",
     "2020 年（与 LST 不同期）",
     "文件名含 precip_sum_2020；血缘未给",
     "约 5 km 粗网格；NaN 表示缺测"),
    ("soil_moist_mean_2020.tif", "土壤水分 MEAN", "预测变量",
     "2020 年（与 LST 不同期）",
     "文件名含 soil_moist_mean_2020；血缘未给",
     "约 10 km 粗网格；NaN 表示缺测"),
]

INFO = {}
records = []
for fname, cname, role, when, source, note in FILES:
    path = gd.RAW / fname
    with rasterio.open(path) as ds:
        arr = ds.read(1).astype("float64")
        nodata = ds.nodata
        if nodata is not None:
            arr[arr == nodata] = np.nan
        arr[~np.isfinite(arr)] = np.nan
        valid = np.isfinite(arr)
        vals = arr[valid]
        res_text = (f"{abs(ds.transform.a):.6f}°"
                    if ds.crs is not None and ds.crs.to_epsg() == 4326
                    else f"{abs(ds.transform.a):.2f} m")
        extent = gd.extent_of(ds.transform, (ds.height, ds.width))
        INFO[fname] = {
            "shape": (ds.height, ds.width),
            "transform": ds.transform,
            "crs": ds.crs,
            "nodata": nodata,
            "extent": extent,
            "dtype": ds.dtypes[0],
            "valid": valid,
            "values": vals,
            "bounds": tuple(ds.bounds),
            "descriptions": ds.descriptions,
            "units": ds.units,
            "tags": ds.tags(),
            "band_tags": ds.tags(1),
            "path": path,
        }
        records.append({
            "文件": fname,
            "变量": cname,
            "角色": role,
            "原始宽度": ds.width,
            "原始高度": ds.height,
            "像元大小": res_text,
            "坐标系": ds.crs.to_string() if ds.crs else "未定义",
            "NoData 声明": "未声明" if nodata is None else f"已声明（{nodata:.4g}）",
            "有效像元": int(valid.sum()),
            "缺测/非有限像元": int((~valid).sum()),
            "最小值": round(float(vals.min()), 4),
            "中位数": round(float(np.median(vals)), 4),
            "最大值": round(float(vals.max()), 4),
            "均值": round(float(vals.mean()), 4),
            "来源（项目说明）": source,
            "时间（仅到可核实粒度）": when,
            "单位": "元数据未注明；图件写“原始像元值”",
            "备注": note,
        })

dictionary = pd.DataFrame(records)
dictionary.to_csv(gd.TABLES / "01_data_dictionary.csv", index=False, encoding="utf-8-sig")

show_cols = ["文件", "变量", "原始宽度", "原始高度", "像元大小", "坐标系",
             "NoData 声明", "有效像元", "最小值", "最大值"]
print(dictionary[show_cols].to_string(index=False))
print("\n数据字典已保存：outputs/tables/01_data_dictionary.csv")

## 1.3　它到底是哪个“甘南”？

先停一下，确认地名。遥感数据很容易因为同名地区拿错地方，所以我们不凭文件名猜，而是打开边界属性，再用坐标和面积交叉核对。


In [ ]:
boundary = gpd.read_file(gd.RAW / "GanNan.shp")
b4326 = boundary.to_crs(4326)
gb = str(int(boundary["gb"].iloc[0]))
area_km2 = float(boundary.to_crs(32652).area.iloc[0] / 1e6)
centroid = boundary.to_crs(32652).geometry.centroid.to_crs(4326).iloc[0]
left, bottom, right, top = b4326.total_bounds

admin = pd.DataFrame([{
    "name": boundary["name"].iloc[0],
    "行政区划代码": gb,
    "六位代码": gb.zfill(9)[-6:],
    "边界范围": f"{left:.4f}–{right:.4f}°E, {bottom:.4f}–{top:.4f}°N",
    "面积（km²）": round(area_km2, 1),
    "质心": f"{centroid.x:.3f}°E, {centroid.y:.3f}°N",
}])
admin.to_csv(gd.TABLES / "01_admin_check.csv", index=False, encoding="utf-8-sig")
print(admin.to_string(index=False))
print("\n地名核对：name = 甘南县；六位代码 = 230225（黑龙江省齐齐哈尔市甘南县）。")
print("甘肃甘南藏族自治州的代码是 623000，二者不是同一地点。")

边界核对的结果是：**甘南县，黑龙江省齐齐哈尔市**，代码 `230225`，边界范围 122.902–124.474°E、47.586–48.530°N，面积约 **4813.8 km²**。目标栅格 `GNLST.tif` 的覆盖范围与它几乎重合，说明这六张图确实围绕同一个县，而不是把甘肃甘南的数据误贴过来。

接下来看“同一地方”是否等于“同一张网格”。从登记表已经能看出：LST 是 176×106，DEM 是 4660×4236，NDVI 是 701×422，气温 189×115，降水 36×22，土壤水分只有 18×12。它们覆盖同一片土地，但像元大小、原点、行列数都不同。

## 1.4　有效像元：有多少像元真的能用？

一张图覆盖了县域，不代表每个像元都有有效观测。比如目标 LST 在县界外是 NoData，DEM 原图还覆盖了不少县外区域，NDVI 则用 0 填了边界外。

接下来我们分别数一数有效像元、缺测和数值范围，再专门确认 NDVI 的 0 在县内还是县外。这里多看一眼，能避免后面把填充值当成真实植被指数。


In [ ]:
stats_rows = []
for fname, cname, role, when, source, note in FILES:
    info = INFO[fname]
    h, w = info["shape"]
    vals = info["values"]
    valid_n = int(info["valid"].sum())
    zeros = int(np.isclose(vals, 0).sum())
    stats_rows.append({
        "文件": fname,
        "变量": cname,
        "总像元": h * w,
        "有效像元": valid_n,
        "有效占比": round(valid_n / (h * w), 4),
        "缺测像元": int(h * w - valid_n),
        "缺测占比": round(1 - valid_n / (h * w), 4),
        "整值 0 个数": zeros if fname == "GNNDVI.tif" else "—",
        "数值范围": f"{vals.min():.4g} – {vals.max():.4g}",
        "中位数": round(float(np.median(vals)), 4),
    })

valid_stats = pd.DataFrame(stats_rows)
valid_stats.to_csv(gd.TABLES / "01_valid_stats.csv", index=False, encoding="utf-8-sig")
print(valid_stats.to_string(index=False))

# NDVI 的 0 是否在县界内？用边界栅格化后逐像元判断。
with rasterio.open(gd.RAW / "GNNDVI.tif") as ds:
    ndvi = ds.read(1).astype("float64")
    inside = ~geometry_mask(boundary.geometry, out_shape=ndvi.shape,
                            transform=ds.transform, invert=False)
zero = np.isclose(ndvi, 0)
zeros_inside = int((zero & inside).sum())
zeros_outside = int((zero & ~inside).sum())
print(f"\nNDVI 县界内像元 {int(inside.sum()):,} 个，其中整值 0：{zeros_inside} 个")
print(f"NDVI 县界外像元 {int((~inside).sum()):,} 个，其中整值 0：{zeros_outside:,} 个"
      f"（占县界外的 {zeros_outside / (~inside).sum():.1%}）")
print(f"NDVI 县界内取值范围：{ndvi[inside].min():.4f} – {ndvi[inside].max():.4f}")

# 用县域面积和 LST 有效像元数交叉核对网格面积
lst_info = INFO["GNLST.tif"]
cell_km2 = (abs(lst_info["transform"].a) * 111.32 * np.cos(np.deg2rad(centroid.y)) *
            abs(lst_info["transform"].e) * 110.57)
print(f"\nLST 像元名义面积约 {cell_km2:.3f} km²（0.008983° 在 48°N 不是正方形公里格）")
print(f"LST 有效像元 × 像元面积 = {lst_info['valid'].sum() * cell_km2:.1f} km²；"
      f"行政边界面积 = {area_km2:.1f} km²")

## 一起来读这些统计结果

先看目标图。7156 个 LST 像元落在县界内，占整张矩形栅格的 38.4%。空出来的 11500 个像元主要是县界外，不是数据缺了一大片。再看 DEM，它原始图幅很大，县外区域也包含在内，所以“有效比例低”不能直接理解为县内数据质量差。

NDVI 这里最值得停下来：181132 个 0 全在县界外，县内一个都没有。换句话说，这些 0 是边界填充值，不能当成真实的“无植被”。降水和土壤水分的有效像元数看起来少，是因为它们原始分辨率很粗，一个像元覆盖的地面范围更大。

最后记住单位问题。数值看起来像温度、降水量或土壤水分，并不能证明单位。源文件没有写清楚，我们就诚实地称它们为“原始像元值”。


## 1.5　先看原生网格长什么样

现在打开六张原始图。这里故意不做重采样，每张都按自己的坐标和分辨率显示。请边看图边留意范围、像元大小和缺测位置。

读图时记住这几个约定：

- 浅灰 `#e4e4e4` 表示缺测或边界外填充；
- DEM 用 UTM 52N，坐标轴和比例尺以 km 表示，其余图用经纬度；
- NDVI 的边界外 0 只在绘图时显示为灰色，原始数据不改；
- 色标统一写“原始像元值”，不擅自补上 °C、mm 或 %。


In [ ]:
def load_native(fname):
    with rasterio.open(gd.RAW / fname) as ds:
        arr = ds.read(1).astype("float64")
        if ds.nodata is not None:
            arr[arr == ds.nodata] = np.nan
        arr[~np.isfinite(arr)] = np.nan
        extent = gd.extent_of(ds.transform, (ds.height, ds.width))
        return arr, extent, ds.crs

panel_spec = {
    "GNLST.tif": dict(letter="(a)", cmap="magma", vmin=6, vmax=18,
                      title="LST 目标变量｜176×106，0.008983°",
                      cbar="LST（原始像元值）"),
    "GNDEM.tif": dict(letter="(b)", cmap="terrain", vmin=76, vmax=365,
                      title="DEM｜4660×4236，26.7 m（UTM 52N）",
                      cbar="高程（原始像元值）"),
    "GNNDVI.tif": dict(letter="(c)", cmap="YlGn", vmin=-0.08, vmax=0.60,
                       title="NDVI｜701×422，0.002246°（0 填充置灰）",
                       cbar="NDVI（原始像元值）"),
    "GNTEM.tif": dict(letter="(d)", cmap="RdYlBu_r", vmin=2.9, vmax=4.9,
                      title="气温 TEM｜189×115，0.008332°",
                      cbar="气温（原始像元值）"),
    "precip_sum_2020.tif": dict(letter="(e)", cmap="YlGnBu", vmin=590, vmax=740,
                                title="年降水｜36×22，0.044916°",
                                cbar="降水（原始像元值）"),
    "soil_moist_mean_2020.tif": dict(letter="(f)", cmap="BrBG", vmin=11.5, vmax=17,
                                     title="土壤水分｜18×12，0.089832°",
                                     cbar="土壤水分（原始像元值）"),
}

fig, axes = plt.subplots(3, 2, figsize=(15.8, 12.8))
fig.subplots_adjust(left=0.055, right=0.975, top=0.925, bottom=0.055,
                    hspace=0.34, wspace=0.20)

for ax, (fname, _, _, _, _, _) in zip(axes.ravel(), FILES):
    spec = panel_spec[fname]
    arr, extent, crs = load_native(fname)
    plot_arr = arr.copy()
    if fname == "GNNDVI.tif":
        plot_arr[np.isclose(plot_arr, 0)] = np.nan   # 仅显示置灰，不修改原始数据
    is_utm = crs is not None and crs.to_epsg() == 32652
    gs.plot_raster(
        ax, plot_arr, extent, spec["cmap"], spec["vmin"], spec["vmax"],
        title=f'{spec["letter"]} {spec["title"]}', cbar_label=spec["cbar"],
        boundary=boundary, crs=crs, add_bar=not is_utm,
    )
    if is_utm:
        gs.add_north_arrow(ax)
        x0 = extent[0] + 0.06 * (extent[1] - extent[0])
        y0 = extent[2] + 0.075 * (extent[3] - extent[2])
        ax.plot([x0, x0 + 20000], [y0, y0], color=gs.INK, lw=2.4,
                solid_capstyle="butt", zorder=20)
        ax.plot([x0, x0], [y0 - 1800, y0 + 1800], color=gs.INK, lw=1.0, zorder=20)
        ax.plot([x0 + 20000, x0 + 20000], [y0 - 1800, y0 + 1800],
                color=gs.INK, lw=1.0, zorder=20)
        ax.text(x0 + 10000, y0 + 3200, "20 km", ha="center", va="bottom",
                fontsize=8.5, color=gs.INK, zorder=20)

fig.suptitle("图 1-1　甘南县六张栅格的原生网格（浅灰为缺测/边界外填充）",
             fontsize=15.5, y=0.975)
fig.text(0.5, 0.012,
         "坐标：DEM 为 UTM 52N（km），其余为 WGS84 经纬度；色标单位均为原始像元值。"
         "NDVI 的 0 为县界外填充，显示时置灰；边界线来自 GanNan.shp。",
         ha="center", va="bottom", fontsize=9.2, color=gs.INK)
gs.save(fig, "01_data_preview.png")
plt.show()
plt.close(fig)
print("已保存：outputs/figures/01_data_preview.png")

六张图放在一起，第一眼的判断应该是：

1. **范围对得上**：六张图的边界线都落在县域轮廓上，LST 的暖色像元严格限制在县界内；DEM 的图幅比县大，但县界清楚可见。
2. **像元质感完全不同**：LST 与 TEM 是较粗的方块，NDVI 更细，降水和土壤水分则是明显的大色块。后面若把它们放进同一个模型，每个 1 km 格子里其实塞进了不同数量的源像元。
3. **缺测位置不同**：LST 的浅灰是县界外，NDVI 的浅灰是 0 填充，降水/土壤水分的浅灰是 NaN。三种“灰”含义不同，不能混为一谈。
4. **DEM 与其它图不同坐标系**：它的坐标轴是 UTM 公里，不是经纬度；这也是它必须单独重投影的原因。

图形也暴露了一个限制：由于各图范围不同，直接按“整幅图”比较会看到大量空白。真正的建模只在 LST 的 7156 个有效像元上进行。

## 1.6　先看看目标变量 LST

建模前，我们先认识一下要预测的量。看左边的地图，哪些地方偏暖、哪些地方偏冷？再看右边直方图，数值主要集中在哪一段，有没有很长的尾巴？这些观察能帮我们判断后续需要关注什么。


In [ ]:
lst = INFO["GNLST.tif"]["values"]
lst_valid = lst[np.isfinite(lst)]
lst_mean = float(lst_valid.mean())
lst_median = float(np.median(lst_valid))
lst_missing = int(INFO["GNLST.tif"]["shape"][0] * INFO["GNLST.tif"]["shape"][1] - lst_valid.size)

fig = plt.figure(figsize=(13.4, 5.9))
gspec = fig.add_gridspec(1, 2, width_ratios=[1.02, 1.0], wspace=0.16,
                         left=0.055, right=0.975, top=0.88, bottom=0.13)
ax0 = fig.add_subplot(gspec[0, 0])
arr, extent, crs = load_native("GNLST.tif")
gs.plot_raster(ax0, arr, extent, "magma", 6, 18,
               title="(a) LST 空间分布（2021 年）",
               cbar_label="LST（原始像元值）",
               boundary=boundary, crs=crs, add_bar=True)
ax0.text(0.025, 0.975, f"有效 {lst_valid.size:,}｜缺测 {lst_missing:,}",
         transform=ax0.transAxes, ha="left", va="top", fontsize=9.2,
         color=gs.INK, zorder=25,
         bbox=dict(boxstyle="round,pad=0.28", fc="white", ec="none", alpha=0.82))

ax1 = fig.add_subplot(gspec[0, 1])
ax1.hist(lst_valid, bins=30, color="#b23a48", edgecolor="white", linewidth=0.6)
ax1.axvline(lst_mean, color="#1f2937", lw=1.4, ls="--", label=f"均值 {lst_mean:.2f}")
ax1.axvline(lst_median, color="#2a6f97", lw=1.4, ls=":", label=f"中位数 {lst_median:.2f}")
ax1.set_xlabel("LST（原始像元值）")
ax1.set_ylabel("像元数")
ax1.set_title("(b) 7156 个有效像元的分布")
ax1.grid(axis="y", color=gs.GRID, lw=0.6, alpha=0.8)
ax1.set_axisbelow(True)
ax1.legend(loc="upper right")
ax1.text(0.975, 0.62,
         f"n = {lst_valid.size:,}\n标准差 = {lst_valid.std(ddof=1):.2f}\n"
         f"范围 = {lst_valid.min():.2f} – {lst_valid.max():.2f}",
         transform=ax1.transAxes, ha="right", va="top", fontsize=9.2,
         color=gs.INK, bbox=dict(boxstyle="round,pad=0.3", fc="#f7f7f7", ec="none"))

fig.suptitle("图 1-2　甘南县 LST 的空间分布与数值分布", fontsize=15, y=0.965)
gs.save(fig, "01_lst_distribution.png")
plt.show()
plt.close(fig)
print("已保存：outputs/figures/01_lst_distribution.png")

LST 的 7156 个有效像元均值 **12.12**、中位数 **11.87**，标准差约 **1.78**，范围 **6.24–17.88**。分布是单峰、略右偏的：大部分像元集中在 10–13，少数高温像元把右尾拉长。这个量级与“地表温度”的常见物理范围相近，但文件没有给单位，不能据此写成 °C。

空间上，LST 并不均匀：南部和西部偏暖，北部与东北部边缘偏冷，中部还有几处冷斑。至于这些差异是地形、植被还是天气造成的，本章不能回答——六张图还没有对齐，任何“看图说话”都只是猜测。下一章先把它们放到同一张网格上，才谈得上比较。

## 1.7　单位和时间，哪些有证据？

读完空间分布，再核对属性信息。请把下面的表当成这套数据的“已知与未知”：能从文件或来源核实的就记录下来，查不到的不要靠数值范围猜。教程后面会沿用这些限制。


In [ ]:
time_hint = {
    "GNLST.tif": "项目说明：来自 LST_Day2021（2021 年）",
    "GNDEM.tif": "无（静态地形）",
    "GNNDVI.tif": "无",
    "GNTEM.tif": "项目说明：来自 TEM2021（2021 年）",
    "precip_sum_2020.tif": "文件名：2020 年",
    "soil_moist_mean_2020.tif": "文件名：2020 年",
}
verdict = {
    "GNLST.tif": "仅年份可核实；月日未知；单位不可核实",
    "GNDEM.tif": "静态量；快照时间与单位不可核实",
    "GNNDVI.tif": "时间不可核实；无 NoData 声明",
    "GNTEM.tif": "仅年份可核实；时间聚合方式未知；单位不可核实",
    "precip_sum_2020.tif": "年份来自文件名；统计期与单位不可核实",
    "soil_moist_mean_2020.tif": "年份来自文件名；统计期与单位不可核实",
}

audit_rows = []
with rasterio.open(gd.RAW / "GNLST.tif") as ds_t:
    target_bounds = tuple(ds_t.bounds)
    target_crs = ds_t.crs
for fname, cname, role, when, source, note in FILES:
    info = INFO[fname]
    desc = info["descriptions"][0] if info["descriptions"] and info["descriptions"][0] else "无"
    unit = info["units"][0] if info["units"] and info["units"][0] else "未注明"
    src_bounds_4326 = transform_bounds(
        info["crs"], "EPSG:4326", *info["bounds"], densify_pts=21
    )
    covers = (src_bounds_4326[0] <= target_bounds[0] and
              src_bounds_4326[1] <= target_bounds[1] and
              src_bounds_4326[2] >= target_bounds[2] and
              src_bounds_4326[3] >= target_bounds[3])
    audit_rows.append({
        "文件": fname,
        "变量": cname,
        "波段描述": desc,
        "元数据单位": unit,
        "元数据时间字段": "无",
        "时间线索": time_hint[fname],
        "可核实程度": verdict[fname],
        "源范围覆盖目标范围": "是" if covers else "否（边缘可能缺测）",
    })

time_audit = pd.DataFrame(audit_rows)
time_audit.to_csv(gd.TABLES / "01_time_unit_audit.csv", index=False, encoding="utf-8-sig")
print(time_audit[["文件", "波段描述", "元数据单位", "元数据时间字段",
                  "时间线索", "源范围覆盖目标范围"]].to_string(index=False))

readiness = pd.DataFrame([
    ("同一行政范围", "是", "六张图都覆盖甘南县；边界面积 4813.8 km²"),
    ("同一坐标系", "否", "DEM 为 EPSG:32652，其余为 EPSG:4326"),
    ("同一网格", "否", "尺寸 176×106 至 4660×4236；间距 26.7 m 至约 10 km"),
    ("同一 NoData 约定", "否", "LST/TEM 为 -3.4e38，DEM 为 -32768，NDVI 未声明且 0 填充，降水/土壤水分为 NaN"),
    ("单位可核实", "否", "六张图的波段单位均为空"),
    ("时间一致", "否", "LST/TEM 为 2021 年，降水/土壤水分为 2020 年，NDVI 无时间"),
    ("能否直接叠成一张表", "否", "必须先重投影、重采样并统一 NoData；见第 2 章"),
])
readiness.columns = ["检查项", "结论", "依据"]
readiness.to_csv(gd.TABLES / "01_model_readiness.csv", index=False, encoding="utf-8-sig")
print("\n" + readiness.to_string(index=False))

### 能不能一起建模？

**能，但不是现在这样直接叠。** 六张图确实覆盖同一个县，空间上具备“一起建模”的前提；可是它们不在同一张网格上：DEM 是 UTM 投影、26.7 m，NDVI 约 250 m，气温约 0.0083°，降水和土壤水分是 5–10 km 的粗网格，而目标是 0.008983° 的经纬度格网。直接按数组下标相加，等于把不同地点、不同大小的像元硬凑在一起，结果没有地理意义。

第二层问题是属性不一致：单位未注明；LST/TEM 只知 2021 年，降水/土壤水分是 2020 年，NDVI 没有时间血缘。即使网格对齐，时间错位也会让“气温解释 LST”这类解释变得含糊。因此本课把任务定位为**留出区域估计的方法演示**，不是甘南县 LST 产品，也不做因果解释。

第三层是缺失机制：LST 的 NoData 主要在县界外，NDVI 的 0 是边界外填充，DEM/TEM 在目标区边缘还会有少量缺测。把它们统一成 `NaN` 之后，才能谈“共同有效像元”。

## 地理视角：为什么这些细节重要？

甘南县位于松嫩平原西缘，地形、水分和植被可能一起影响地表温度，所以变量之间很可能相关。但我们现在还不能据这几张图讲机制，数据的空间尺度和时间还没对齐。

也提醒大家，“约 1 km”只是对纬度方向分辨率的口头简称。在 48°N，经纬度网格东西向约 0.67 km，南北向约 1 km。下一章要把栅格放到同一参考网格上，这个几何差别会一直跟着我们。


In [ ]:
runtime_sec = time.time() - T0
summary = {
    "notebook": "notebooks/01_认识数据.ipynb",
    "kernel": "geoml",
    "python": sys.executable,
    "conda_prefix": os.environ.get("CONDA_PREFIX"),
    "run_started": RUN_START.isoformat(timespec="seconds"),
    "runtime_sec": round(runtime_sec, 1),
    "question": "六张栅格能不能一起建模？",
    "study_area": {
        "name": "甘南县",
        "admin": "黑龙江省齐齐哈尔市",
        "gb_code": "230225",
        "bounds_4326": [round(float(v), 4) for v in (left, bottom, right, top)],
        "area_km2": round(area_km2, 1),
    },
    "target_grid": {
        "file": "GNLST.tif",
        "shape": [106, 176],
        "crs": "EPSG:4326",
        "res_deg": 0.008983152841195198,
        "valid_pixels": int(INFO["GNLST.tif"]["valid"].sum()),
        "missing_pixels": int(106 * 176 - INFO["GNLST.tif"]["valid"].sum()),
        "valid_fraction": round(float(INFO["GNLST.tif"]["valid"].sum()) / (106 * 176), 4),
    },
    "key_numbers": {
        "lst_mean": round(lst_mean, 4),
        "lst_median": round(lst_median, 4),
        "lst_std": round(float(lst_valid.std(ddof=1)), 4),
        "lst_min": round(float(lst_valid.min()), 4),
        "lst_max": round(float(lst_valid.max()), 4),
        "dem_valid_pixels": int(INFO["GNDEM.tif"]["valid"].sum()),
        "ndvi_zero_total": int(np.isclose(INFO["GNNDVI.tif"]["values"], 0).sum()),
        "ndvi_zero_inside_boundary": zeros_inside,
        "ndvi_zero_outside_boundary": zeros_outside,
        "precip_valid_pixels": int(INFO["precip_sum_2020.tif"]["valid"].sum()),
        "soil_valid_pixels": int(INFO["soil_moist_mean_2020.tif"]["valid"].sum()),
        "lst_pixel_area_km2": round(cell_km2, 3),
    },
    "figures": [
        "outputs/figures/01_data_preview.png",
        "outputs/figures/01_lst_distribution.png",
    ],
    "tables": [
        "outputs/tables/01_data_dictionary.csv",
        "outputs/tables/01_admin_check.csv",
        "outputs/tables/01_valid_stats.csv",
        "outputs/tables/01_time_unit_audit.csv",
        "outputs/tables/01_model_readiness.csv",
        "outputs/tables/01_run_summary.json",
    ],
    "unverified": [
        "六张图的物理单位均未在 TIFF 标签或波段单位中注明；正文统一写“原始像元值”。",
        "LST 仅知 2021 年，月日未知；TEM 仅知 2021 年，时间聚合方式未知。",
        "降水与土壤水分仅从文件名知 2020 年，统计期定义与血缘未核实。",
        "NDVI 的观测时间与来源未核实；0 填充只在县界外。",
        "DEM 快照时间未核实。",
    ],
    "notes": "本章未做重采样，未改动 data/processed 下的核心产物。",
}
gd.write_json(summary, "01_run_summary.json")
print(f"运行耗时：{runtime_sec:.1f} s")
print("已保存：outputs/tables/01_run_summary.json")

## 本章小结

这一章先把数据认清楚了：研究区是黑龙江省齐齐哈尔市甘南县，六张栅格覆盖同一县域，但 CRS、分辨率、范围和缺测约定都不同。NDVI 的 181132 个零值位于县界外，不能误作县内观测。时间和单位信息也不完整，所以后续把任务限定为留出区域的预测方法演示。

下一章我们会选 LST 作为参考网格，把五个输入变量逐一对齐，并检查重采样有没有引入新的缺测。
